In [1]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error 
from scipy import stats
import sys
import scipy.signal
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *
from adversary import TA_tools
import importlib
importlib.reload(TA_tools)

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "/home/40265864@ecit.qub.ac.uk/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_tolerance             changed from 1144409.1796875           to 13096723.705530167       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx         

In [19]:
scope.adc.samples = 12000                 # Number of samples per segment
scope.adc.stream_mode = "segmented"            # Enable segmented capture
scope.adc.segments = 1000          # Total number of segments to capture  
scope.adc.bits_per_sample = 12
scope.clock.adc_mul = 26
scope.gain.mode = "high"
scope.gain.gain = 15
scope.clock.reset_adc()

In [20]:
%%bash -s "$PLATFORM" "$SS_VER"
cd firmware/XOR_test
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Welcome to another exciting ChipWhisperer target build!!
Size after:
+--------------------------------------------------------
+ Built for platform Microchip SAM4S with:
+ CRYPTO_TARGET = NONE
   text	   data	    bss	    dec	    hex	filename
   6936	    104	   4144	  11184	   2bb0	XOR_test-CW308_SAM4S.elf
+ CRYPTO_OPTIONS = AES128C
+--------------------------------------------------------


In [21]:
cw.program_target(scope, prog, "firmware/XOR_test/XOR_test-{}.hex".format(PLATFORM))

In [22]:
tools = TA_tools.Tools(scope,target)

In [23]:
x0 = "00000000\n"  #00
x1 = "00001000\n"  #01
x2 = "10000000\n"  #10
x3 = "10001000\n"  #11

dummy = "11111111\n"

warmup = tools.get_trace(x0)

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:712) FIFO error occured; see scope.adc.errors for details.


In [24]:
def align_traces(traces):
    ref_trace = traces[0]  # Use the first trace as reference
    aligned_traces = []

    for trace in traces:
        correlation = np.correlate(trace, ref_trace)  # Compute cross-correlation
        shift = np.argmax(correlation) - (len(trace) - 1)  # Find best alignment
        aligned_trace = np.roll(trace, -shift)  # Shift the trace
        aligned_traces.append(aligned_trace)
    
    return np.array(aligned_traces)

In [25]:
warmup = align_traces(warmup)

In [26]:
traces0 = []
traces1 = []
traces2 = []
traces3 = []

for i in range(1000):
    if i%4==0:
        traces0.append(warmup[i])
    if i%4==1:
        traces1.append(warmup[i])
    if i%4==2:
        traces2.append(warmup[i])
    if i%4==3:
        traces3.append(warmup[i])

avg0 = np.mean(traces0,axis=0)
avg1 = np.mean(traces1,axis=0)
avg2 = np.mean(traces2,axis=0)
avg3 = np.mean(traces3,axis=0)

In [27]:
cw.plot(avg0) * cw.plot(avg1) * cw.plot(avg2) * cw.plot(avg3)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)

In [29]:
cw.plot(warmup[50]) * cw.plot(warmup[99])

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [ ]:
def get_trace():
    #num_char = target.in_waiting()
    #while num_char > 0:
        #target.read(num_char, 10)
        #time.sleep(0.01)
        #num_char = target.in_waiting()
    time.sleep(0.1)
    #target.flush()
    scope.arm()
    target.write("Start\n")
    if scope.capture():
        raise RuntimeError("Capture failed")
    trace_segments = scope.get_last_trace_segmented()
    alligned_traces = align_traces(trace_segments)
    return alligned_traces

In [ ]:
reset_target(scope)

traces = get_trace()

In [ ]:
cw.plot(traces[0]) * cw.plot(traces[1]) * cw.plot(traces[2]) * cw.plot(traces[3])

In [ ]:
avg0 = []
avg1 = []
avg2 = []
avg3 = []
for i in range(250):
    avg0.append(traces[int(4*i)])
    avg1.append(traces[int(4*i +1)])    
    avg2.append(traces[int(4*i +2)])    
    avg3.append(traces[int(4*i +3)])
avg0 = np.mean(avg0, axis=0)
avg1 = np.mean(avg1, axis=0)
avg2 = np.mean(avg2, axis=0)
avg3 = np.mean(avg3, axis=0)

In [ ]:
cw.plot(avg0) * cw.plot(avg1) * cw.plot(avg2) * cw.plot(avg3)

In [ ]:
cw.plot(avg2 - avg3)